# T1.3 – Binary Collision Severity Classification (Slight vs. Serious/Fatal)

This notebook reframes the T1.2 problem as binary classification and compares a majority-class baseline, logistic regression, and a deep neural network.

## 1. Setup & Imports

In [ ]:
# Standard data & visualisation libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-learn (machine learning & evaluation)
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_score,
    recall_score,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)

# TensorFlow / Keras (deep learning)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Set the overall visual style for the plots
sns.set_theme(style="whitegrid")
print("Environment setup complete. All libraries imported.")


## 2. Load Data & Preprocessor from T1.2

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load the bundle exported from T1.2
imported_data = joblib.load('/content/drive/MyDrive/Machine Learning Project/task1_2_export.joblib')

# Extract the variables
X = imported_data['X']
y = imported_data['y']
preprocessor = imported_data['preprocessor']

print("Dataset and preprocessor successfully loaded!")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


## 3. Relabel Target for Binary Classification
### Majority-Class Baseline

In [ ]:
# 1. Reformulate the labels for binary classification
# T1.2 labels were: 0=Slight, 1=Serious, 2=Fatal
# T1.3 labels:      0=Slight, 1=Serious/Fatal
y_binary = y.replace({2: 1})

print("Labels remapped for T1.3: 0=Slight, 1=Serious/Fatal")
print(y_binary.value_counts(), "\n")

# 2. Universal binary evaluation function
def evaluate_binary_model(pipeline, X, y, model_name):
    print(f"========== Evaluating {model_name} ==========")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # For binary classification we need probabilities to calculate AUROC and AUPRC.
    # Extract the probabilities for the positive class (index 1).
    probs = cross_val_predict(pipeline, X, y, cv=skf, n_jobs=-1, method='predict_proba')[:, 1]

    # Default threshold of 0.5, as required by the brief
    preds = (probs >= 0.5).astype(int)

    # Calculate required metrics
    acc = accuracy_score(y, preds)
    bal_acc = balanced_accuracy_score(y, preds)
    auroc = roc_auc_score(y, probs)
    auprc = average_precision_score(y, probs)
    cm = confusion_matrix(y, preds)

    print(f"Overall Accuracy:  {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"AUROC:             {auroc:.4f}")
    print(f"AUPRC:             {auprc:.4f}\n")

    print("Classification Report (Threshold = 0.5):")
    print(classification_report(y, preds, target_names=['0 (Slight)', '1 (Serious/Fatal)'], labels=[0, 1]))

    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=['True Slight', 'True Serious/Fatal'],
        columns=['Pred Slight', 'Pred Serious/Fatal']
    ))
    print("========================================================\n")

# 3. Implement the majority-class baseline
# DummyClassifier predicts the most frequent class (0 = Slight)
majority_pipeline_binary = Pipeline(steps=[
    ('preprocessor', preprocessor), # Re-using the same ColumnTransformer
    ('classifier', DummyClassifier(strategy='prior'))
])

evaluate_binary_model(majority_pipeline_binary, X, y_binary, "Binary Majority-Class Baseline")


## 4. Binary Logistic Regression

In [ ]:
print("Training Binary Logistic Regression (this will take a moment)...")

# 1. Define the pipeline
logreg_binary_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',  # Crucial to handle the 75/25 split
        max_iter=1000,            # High iteration cap to ensure convergence
        n_jobs=-1,
        random_state=42
    ))
])

# 2. Evaluate using the custom binary evaluation function
evaluate_binary_model(logreg_binary_pipeline, X, y_binary, "Binary Logistic Regression")


## 5. Binary Deep Neural Network (DNN)

In [ ]:
print("Preprocessing dataset for the neural network...")
# The DNN requires the data to be fully transformed into a NumPy array first
X_processed = preprocessor.fit_transform(X)

# 1. Calculate binary class weights
weights = compute_class_weight('balanced', classes=np.unique(y_binary), y=y_binary)
binary_class_weights_dict = {0: weights[0], 1: weights[1]}
print(f"Applied Binary Class Weights: {binary_class_weights_dict}\n")

# 2. K-fold cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
binary_dnn_oof_probs = np.zeros(len(y_binary))

print("Training Binary DNN across 5 folds (this will take a few minutes)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y_binary)):
    print(f"--- Training Fold {fold + 1} ---")

    # Split the data
    X_train, X_val = X_processed[train_idx], X_processed[val_idx]
    y_train, y_val = y_binary.iloc[train_idx], y_binary.iloc[val_idx]

    # Build the binary architecture
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')  # Single output node with sigmoid for binary classification
    ])

    # Binary crossentropy loss, matching the sigmoid output
    model.compile(optimizer='adam', loss='binary_crossentropy')

    # Define early stopping
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    # Train the model
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=256,
        class_weight=binary_class_weights_dict,
        callbacks=[early_stopping],
        verbose=0 # Suppresses the epoch-by-epoch output
    )

    # Extract probabilities (flattened to a 1D array)
    probs = model.predict(X_val).flatten()
    binary_dnn_oof_probs[val_idx] = probs

# 3. Evaluate the DNN at the default 0.5 threshold
preds = (binary_dnn_oof_probs >= 0.5).astype(int)

acc = accuracy_score(y_binary, preds)
bal_acc = balanced_accuracy_score(y_binary, preds)
auroc = roc_auc_score(y_binary, binary_dnn_oof_probs)
auprc = average_precision_score(y_binary, binary_dnn_oof_probs)
cm = confusion_matrix(y_binary, preds)

print("\n========== Evaluating Binary Deep Neural Network ==========")
print(f"Overall Accuracy:  {acc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"AUROC:             {auroc:.4f}")
print(f"AUPRC:             {auprc:.4f}\n")

print("Classification Report (Threshold = 0.5):")
print(classification_report(y_binary, preds, target_names=['0 (Slight)', '1 (Serious/Fatal)'], labels=[0, 1]))

print("Confusion Matrix:")
print(pd.DataFrame(
    cm,
    index=['True Slight', 'True Serious/Fatal'],
    columns=['Pred Slight', 'Pred Serious/Fatal']
))
print("=========================================================\n")


## 6. Precision-Recall Curve Comparison

In [ ]:
print("Extracting Logistic Regression probabilities for the plot...")
# Extract LogReg probabilities for the positive class (1)
logreg_probs = cross_val_predict(
    logreg_binary_pipeline,
    X,
    y_binary,
    cv=skf,
    n_jobs=-1,
    method='predict_proba'
)[:, 1]

# Calculate precision and recall arrays for both models
lr_precision, lr_recall, lr_thresholds = precision_recall_curve(y_binary, logreg_probs)
dnn_precision, dnn_recall, dnn_thresholds = precision_recall_curve(y_binary, binary_dnn_oof_probs)

# Set up the visualisation
plt.figure(figsize=(10, 7))

# Plot both models
plt.plot(lr_recall, lr_precision, label='Logistic Regression', color='blue', linewidth=2)
plt.plot(dnn_recall, dnn_precision, label='Deep Neural Network', color='orange', linewidth=2)

# Add the baseline (proportion of the positive class in the dataset)
baseline_ratio = y_binary.mean()
plt.axhline(y=baseline_ratio, color='red', linestyle='--', label=f'Baseline ({baseline_ratio:.3f})')

# Formatting
plt.title('Binary Classification: Precision-Recall Curve (Slight vs. Serious/Fatal)', fontsize=14)
plt.xlabel('Recall (True Positive Rate)', fontsize=12)
plt.ylabel('Precision (Positive Predictive Value)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


## 7. Generate Predictions for Visual Comparison

In [ ]:
print("Generating predictions for visual comparison...")

# 1. Reuse the positive class probabilities extracted in Section 6
logreg_binary_probs = logreg_probs

# 2. Get the predictions at the 0.5 threshold for both models
logreg_binary_preds = (logreg_binary_probs >= 0.5).astype(int)
binary_dnn_preds = (binary_dnn_oof_probs >= 0.5).astype(int)

### Confusion Matrix Comparison (Threshold = 0.5)

In [ ]:
print("Generating confusion matrix comparison heatmaps...")

# 1. Calculate the matrices using the 0.5 threshold predictions
cm_logreg_binary = confusion_matrix(y_binary, logreg_binary_preds)
cm_dnn_binary = confusion_matrix(y_binary, binary_dnn_preds)

# 2. Set up the figure as a 1x2 grid
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Labels for the axes
labels_binary = ['Slight (0)', 'Serious/Fatal (1)']

# 3. Plot the Logistic Regression heatmap
sns.heatmap(cm_logreg_binary, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=labels_binary, yticklabels=labels_binary, cbar=False)
axes[0].set_title('Binary Logistic Regression (Thresh=0.5)', fontsize=14)
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# 4. Plot the Deep Neural Network heatmap
# A different colour palette ('Oranges') makes the visual comparison easier
sns.heatmap(cm_dnn_binary, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=labels_binary, yticklabels=labels_binary, cbar=False)
axes[1].set_title('Binary Deep Neural Network (Thresh=0.5)', fontsize=14)
axes[1].set_xlabel('Predicted Label', fontsize=12)

plt.suptitle('Confusion Matrix Comparison: Binary Classification', fontsize=16)
plt.tight_layout()
plt.show()


## 8. ROC Curve Comparison

In [ ]:
print("Generating ROC curve plot...")

# 1. Calculate the ROC arrays for both models and their matching AUROC values
lr_fpr, lr_tpr, _ = roc_curve(y_binary, logreg_binary_probs)
lr_auc = roc_auc_score(y_binary, logreg_binary_probs)

dnn_fpr, dnn_tpr, _ = roc_curve(y_binary, binary_dnn_oof_probs)
dnn_auc = roc_auc_score(y_binary, binary_dnn_oof_probs)

# 2. Set up the visualisation
plt.figure(figsize=(10, 7))

# 3. Plot both curves
plt.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUROC = {lr_auc:.4f})', color='blue', linewidth=2)
plt.plot(dnn_fpr, dnn_tpr, label=f'Deep Neural Network (AUROC = {dnn_auc:.4f})', color='orange', linewidth=2)

# 4. Plot the diagonal line representing random guessing (0.5 AUROC baseline)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guess (Baseline)')

# 5. Finalise the formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.title('Receiver Operating Characteristic (ROC) Curves', fontsize=14)
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


## 9. Final Performance Summary Table

In [ ]:
# Compile the final binary metrics gathered above into a dictionary
summary_data_t13 = {
    'Model': [
        'Majority-Class Baseline',
        'Binary Logistic Regression',
        'Deep Neural Network (DNN)'
    ],
    'Overall Accuracy': [0.7516, 0.5874, 0.6270],
    'Balanced Accuracy': [0.5000, 0.5767, 0.5796],
    'AUROC': [0.5000, 0.6050, 0.6109],
    'AUPRC': [0.2484, 0.3407, 0.3457]
}

# Create the DataFrame
summary_table_t13 = pd.DataFrame(summary_data_t13)

# Print the formatted table
print("Task 1.3: Binary Model Performance Summary Table")
print("-" * 80)
print(summary_table_t13.to_string(index=False, justify='center'))
print("-" * 80)


## 10. Custom Decision Threshold Evaluation (0.3)

In [ ]:
# Custom threshold evaluation

custom_threshold = 0.3  # Adjustable (e.g. 0.25, 0.3, 0.4) to see how the metrics shift

print(f"========================================================")
print(f" EVALUATING MODELS AT CUSTOM THRESHOLD = {custom_threshold}")
print(f"========================================================")

for name, probs in [("Binary Logistic Regression", logreg_binary_probs),
                    ("Deep Neural Network", binary_dnn_oof_probs)]:

    # Apply the custom threshold
    preds_custom = (probs >= custom_threshold).astype(int)

    # Calculate key metrics
    prec = precision_score(y_binary, preds_custom)
    rec = recall_score(y_binary, preds_custom)
    acc = accuracy_score(y_binary, preds_custom)
    bal_acc = balanced_accuracy_score(y_binary, preds_custom)
    cm = confusion_matrix(y_binary, preds_custom)

    print(f"\n--- {name} ---")
    print(f"Overall Accuracy:          {acc:.4f}")
    print(f"Balanced Accuracy:         {bal_acc:.4f}")
    print(f"Precision (Serious/Fatal): {prec:.4f}")
    print(f"Recall    (Serious/Fatal): {rec:.4f}")
    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=['True Slight', 'True Serious/Fatal'],
        columns=['Pred Slight', 'Pred Serious/Fatal']
    ))
print("========================================================")


### Confusion Matrix Comparison (Threshold = 0.3)

In [ ]:
# Visualise confusion matrices at the custom threshold (0.3)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
labels_binary = ['Slight (0)', 'Serious/Fatal (1)']

for i, (name, probs) in enumerate([("Logistic Regression (Thresh=0.3)", logreg_binary_probs),
                                  ("Deep Neural Network (Thresh=0.3)", binary_dnn_oof_probs)]):

    preds_custom = (probs >= 0.3).astype(int)
    cm_custom = confusion_matrix(y_binary, preds_custom)

    # Select colour maps to match the previous style
    cmap_choice = 'Blues' if i == 0 else 'Oranges'

    sns.heatmap(cm_custom, annot=True, fmt='d', cmap=cmap_choice, ax=axes[i],
                xticklabels=labels_binary, yticklabels=labels_binary, cbar=False)
    axes[i].set_title(name, fontsize=14)
    axes[i].set_ylabel('True Label', fontsize=12)
    axes[i].set_xlabel('Predicted Label', fontsize=12)

plt.suptitle('Confusion Matrix Comparison at Custom Threshold (0.3)', fontsize=16)
plt.tight_layout()
plt.show()
